In [ ]:
# CELL 1: Imports
import os
import random
import shutil
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
# CELL 2: Configurations
import utils.configs

print("Using device:", utils.configs.DEVICE)

In [ ]:
# CELL 3: Inspect raw dataset

data_path = Path(utils.configs.DATA_DIR)

CLASS_NAMES = sorted(
    [folder.name for folder in data_path.iterdir() if folder.is_dir()]
)

NUM_CLASSES = len(CLASS_NAMES)

print("Classes found:", CLASS_NAMES)
print("Number of classes found:", NUM_CLASSES)

total_images = 0

for class_name in CLASS_NAMES:
    class_path = data_path / class_name
    image_count = len(list(class_path.glob("*")))
    total_images += image_count
    print(f"{class_name}: {image_count} images")

print(f"\nTotal images: {total_images}")

# Splitting the Data into Train, Test, Validation 
```
data_split/
    train/
        bike/
        car/
    val/
        bike/
        car/
    test/
        bike/
        car/
```

In [ ]:
# CELL 4: Split dataset safely

from PIL import Image

source_path = Path(utils.configs.DATA_DIR)
split_path = Path(utils.configs.SPLIT_DIR)

if split_path.exists():
    shutil.rmtree(split_path)

def is_valid_image(filepath):
    try:
        with Image.open(filepath) as img:
            img.verify()
        return True
    except:
        return False

for cls in CLASS_NAMES:
    all_files = list((source_path / cls).glob("*"))

    valid_files = []

    for file in all_files:
        if is_valid_image(file):
            valid_files.append(file)
        else:
            print(f"Skipping corrupted file: {file}")

    random.shuffle(valid_files)

    total = len(valid_files)

    train_end = int(total * utils.configs.TRAIN_SPLIT)
    val_end = train_end + int(total * utils.configs.VAL_SPLIT)

    train_files = valid_files[:train_end]
    val_files = valid_files[train_end:val_end]
    test_files = valid_files[val_end:]

    for split_name, file_list in {
        "train": train_files,
        "val": val_files,
        "test": test_files
    }.items():

        target_dir = split_path / split_name / cls
        target_dir.mkdir(parents=True, exist_ok=True)

        for file in file_list:
            shutil.copy(file, target_dir / file.name)

print("Dataset split complete.")

In [ ]:
# CELL 5: Image Transforms
from utils.inference import transform_image, transform_train_image

train_transform = lambda image: transform_train_image(image, utils.configs.IMAGE_SIZE)

test_transform = lambda image: transform_image(image, utils.configs.IMAGE_SIZE)

In [ ]:
# CELL 6: Create the datasets
# Create datasets

train_dataset = datasets.ImageFolder(
    root=Path(utils.configs.SPLIT_DIR) / "train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=Path(utils.configs.SPLIT_DIR) / "val",
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    root=Path(utils.configs.SPLIT_DIR) / "test",
    transform=test_transform
)

print("Class indices:", train_dataset.class_to_idx)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
# CELL 7: Data Loaders
# Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=utils.configs.BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=utils.configs.BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=utils.configs.BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# CELL 8: Visual Checks

images, labels = next(iter(train_loader))
print(images.shape)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, ax in enumerate(axes.flat):
    img = images[i]

    # Convert from [C,H,W] to [H,W,C]
    img = img.permute(1, 2, 0).numpy()

    # Undo normalization for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    img = (img * std) + mean
    img = np.clip(img, 0, 1)

    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels[i].item()])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# CELL 9: Load pretrained resnet50

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Replace final classifier
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# Move model to GPU/CPU
model = model.to(utils.configs.DEVICE)

print(model.fc)
print("Model loaded on:", utils.configs.DEVICE)

In [ ]:
# Freeze entire backbone first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4
for param in model.layer4.parameters():
    param.requires_grad = True

# Unfreeze classifier
for param in model.fc.parameters():
    param.requires_grad = True

print("Layer4 and classifier head trainable.")

In [ ]:
# CELL 11: Loss function and optimizer

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

print("Loss and optimizer configured.")

In [ ]:
# CELL 12: Training funtion
from utils.train import train_one_epoch, validate

In [ ]:
# CELL 13: Main training loop

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

num_epochs = utils.configs.EPOCHS # Change here if you want to fine tune for more epochs after initial training

#load the latest checkpoint from the "checkpoints" folder if it exists
checkpoint_folder = Path("checkpoints")
if checkpoint_folder.exists():
    checkpoint_files = list(checkpoint_folder.glob("model_epoch_*.pth"))
    if checkpoint_files:
        latest_checkpoint = max(checkpoint_files, key=os.path.getctime) # Return the latest checkpoint based on creation time
        print(f"Loading checkpoint: {latest_checkpoint}")
        checkpoint = torch.load(latest_checkpoint, map_location=utils.configs.DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        print("No checkpoints found, starting training from scratch.")
else:
    print("Checkpoint folder not found, starting training from scratch.")

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model = model,
        dataloader = train_loader,
        loss_fn = loss_fn,
        optimizer = optimizer,
        device = utils.configs.DEVICE
    )

    val_loss, val_acc = validate(
        model = model,
        dataloader = val_loader,
        loss_fn = loss_fn,
        device = utils.configs.DEVICE
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    #Save checkpoint to folder named "checkpoints" with filename "model_epoch_{epoch+1}.pth"
    from utils.save_model import save_model
    save_model(
        model=model,
        folder_name="checkpoints",
        model_name=f"model_epoch_{epoch+1}.pth"
    )
    print(f"Checkpoint saved for epoch {epoch+1}.")

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_acc:.2f}%")
    print("-" * 50)

In [ ]:
# CELL 15: Plot training curves

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# CELL 16: Test evaluation

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(utils.configs.DEVICE)
        labels = labels.to(utils.configs.DEVICE)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = 100 * np.mean(np.array(all_predictions) == np.array(all_labels))

print(f"Test Accuracy: {test_accuracy:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_predictions, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_predictions)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot()
plt.show()

In [ ]:
# CELL 17: Save model
from utils.save_model import save_model
save_model(
    model=model,
    folder_name=utils.configs.MODEL_DIR,
    model_name=utils.configs.MODEL_NAME
)